# BioWire 🧠⚡ — Drosophila Visual Circuit SNN

> **Biologically constrained spiking neural network simulation of the *Drosophila melanogaster* optic lobe,  
> built from the FlyWire electron-microscopy connectome.**

[![Python 3.10+](https://img.shields.io/badge/python-3.10%2B-blue)](https://www.python.org/)
[![Brian2](https://img.shields.io/badge/simulator-Brian2-green)](https://brian2.readthedocs.io/)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](LICENSE)
[![Version](https://img.shields.io/badge/version-1.0.0-brightgreen)](CHANGELOG.md)

## Overview

BioWire simulates the **left medulla (ME\_L)** of the *Drosophila* optic lobe using a  
conductance-based leaky integrate-and-fire (LIF) network driven by real connectome data  
(or a synthetic mock when offline). Key features:

- **FlyWire connectome integration** — real synapse counts from codex.flywire.ai  
- **Offline mode** — synthetic mock connectome auto-generated when data files are absent  
- **E/I classification** — 30+ neurotransmitter-typed inhibitory types (DOI-cited)  
- **4-bit signed weight quantisation** — memory-efficient sparse matrices  
- **Parameter optimisation** — grid search calibrated to in-vivo electrophysiology  
- **Three-layer negative control** — rewired / shuffled / random topology comparison  

## Pipeline

| Step | Description |
|------|-------------|
| 1 | Install dependencies & write `environment.yml` |
| 2 | Load FlyWire connectome (or generate mock) |
| 3 | Extract ME\_L circuit, build weight matrices |
| 4 | E/I classification + 4-bit weight quantisation |
| 5 | Network topology analysis |
| 6 | Visualise signed weight matrices |
| 7 | Define conductance-based LIF model (Brian2) |
| 8 | Core 8-type circuit simulation |
| 9 | Full 100-type + mid-tier 20-type simulation |
| 10 | Stimulus–response visualisation + directional tuning |
| 11 | Electrophysiology comparison (sim vs. in-vivo) |
| 12 | Three-layer negative control (Hamming distance) |

## Quick Start

```bash
conda env create -f environment.yml
conda activate biowire
jupyter notebook biowire.ipynb
```

> **Data:** Download `consolidated_cell_types.csv.gz` and  
> `connections_princeton_no_threshold.csv.gz` from [codex.flywire.ai](https://codex.flywire.ai/api/download)  
> and place them in the same directory. Without them the notebook runs in **offline/mock mode**.


## Step 1 — Install Dependencies

In [ ]:
!pip install numpy pandas matplotlib networkx brian2 scipy -q

# ── FIX: Generate environment.yml for reproducibility ──────────────────
import subprocess, sys

_ENV_YML = """name: biowire
channels:
  - conda-forge
  - defaults
dependencies:
  - python=3.10
  - numpy>=1.24
  - pandas>=2.0
  - matplotlib>=3.7
  - networkx>=3.1
  - scipy>=1.11
  - pip:
    - brian2>=2.5.4
"""
with open('environment.yml', 'w') as f:
    f.write(_ENV_YML)
print("✅ Dependencies installed.")
print("✅ environment.yml written — use: conda env create -f environment.yml")

## Step 2 — Load FlyWire Connectome Data

Raw tables from codex.flywire.ai:
- **consolidated_cell_types**: maps neuron `root_id` → `primary_type`
- **connections_princeton_no_threshold**: all synaptic connections with `syn_count`

**V1 FIX:** If the CSV files are absent a synthetic mock connectome is automatically
generated so all downstream cells can still run end-to-end.


In [ ]:
import pandas as pd
import numpy as np
import os, sys, io, warnings
warnings.filterwarnings('ignore')

# ── Global random seed for reproducibility ─────────────────────────────
RNG_SEED   = 42
rng_global = np.random.default_rng(RNG_SEED)

CELL_TYPES_FILE  = 'consolidated_cell_types.csv.gz'
CONNECTIONS_FILE = 'connections_princeton_no_threshold.csv.gz'

# ── V1 FIX: Synthetic mock connectome when real data not available ──────
def _build_mock_connectome(rng, n_neurons=4000, n_edges=80000):
    """
    Generates a biologically plausible mock ME_L connectome for offline testing.
    Neuron-type prevalence roughly mirrors FlyWire ME_L counts.
    """
    # Canonical ME_L types with approximate relative abundances
    type_weights = {
        'L1':120,'L2':115,'L3':60,'L5':55,'Mi1':90,'Mi4':70,'Mi9':50,
        'C2':40,'C3':45,'Tm1':80,'Tm2':75,'Tm4':65,'Tm9':55,'Tm20':40,
        'T4a':35,'T4b':35,'T4c':35,'T4d':35,'T5a':35,'T5b':35,'T5c':35,'T5d':35,
        'Dm1':30,'Dm2':28,'Dm3':28,'Dm4':25,'Dm7':22,'Dm8':22,'Dm9':20,
        'TmY4':18,'TmY5a':18,'TmY10':16,'Am':14,'Pm1':12,'Pm2':12,
        'R7':50,'R8':50,'R1-6':300,
        'Lai':10,'Lawf1':10,'Lawf2':10,'OA-AL2b1':8,'aMe12':8,
    }
    types  = list(type_weights.keys())
    wts    = np.array(list(type_weights.values()), dtype=float)
    wts   /= wts.sum()

    root_ids   = np.arange(1, n_neurons + 1)
    prim_types = rng.choice(types, size=n_neurons, p=wts)
    cell_types = pd.DataFrame({'root_id': root_ids, 'primary_type': prim_types})

    pre_ids  = rng.choice(root_ids, size=n_edges)
    post_ids = rng.choice(root_ids, size=n_edges)
    syn_cnts = rng.integers(1, 120, size=n_edges)
    neuropil = np.full(n_edges, 'ME_L')
    connections = pd.DataFrame({
        'pre_root_id':  pre_ids,
        'post_root_id': post_ids,
        'syn_count':    syn_cnts,
        'neuropil':     neuropil,
    })
    return cell_types, connections

missing = [f for f in [CELL_TYPES_FILE, CONNECTIONS_FILE] if not os.path.exists(f)]
if missing:
    print("⚠️  Missing data file(s):")
    for f in missing:
        print(f"  • {f}")
    print("  Download from: https://codex.flywire.ai/api/download")
    print("  → Using synthetic mock connectome for offline demonstration.\n")
    cell_types, connections = _build_mock_connectome(rng_global)
    USING_MOCK = True
else:
    print("Loading cell type annotations...")
    cell_types = pd.read_csv(CELL_TYPES_FILE)
    print(f"  Neurons annotated : {cell_types.shape[0]:,}")
    print("Loading synaptic connections...")
    connections = pd.read_csv(CONNECTIONS_FILE)
    print(f"  Synaptic edges    : {connections.shape[0]:,}")
    USING_MOCK = False

# ── Validate required columns ───────────────────────────────────────────
REQUIRED_CONN_COLS = {'pre_root_id', 'post_root_id', 'syn_count', 'neuropil'}
REQUIRED_TYPE_COLS = {'root_id', 'primary_type'}
missing_conn = REQUIRED_CONN_COLS - set(connections.columns)
missing_type = REQUIRED_TYPE_COLS - set(cell_types.columns)
if missing_conn:
    raise ValueError(f"connections table missing columns: {missing_conn}\nFound: {list(connections.columns)}")
if missing_type:
    raise ValueError(f"cell_types table missing columns: {missing_type}\nFound: {list(cell_types.columns)}")

print(f"🔒 Random seed fixed to {RNG_SEED} for full reproducibility.")
print(f"{'⚠️  MOCK DATA — for demonstration only' if USING_MOCK else '✅ Real FlyWire data loaded.'}")
print(f"✅ Data files validated.")

## Step 3 — Extract the ME_L (Left Medulla) Circuit

The medulla is the primary motion-processing neuropil of the fly visual system.

**V1 FIX:** Weight matrices now use `scipy.sparse` (CSR format) for memory efficiency.
Dense conversion happens only when needed for Brian2 indexing.

**V1 NOTE — Connectome selection bias:** The top-N type model selects by neuron count
(*abundance bias*). A Gini coefficient and coverage fraction are printed so users can
assess how representative the subset is.

| Model | Neuron types | Purpose |
|---|---|---|
| `core` | 8 | Canonical motion circuit |
| `mid`  | 50 | Extended medulla coverage |
| `full` | 100 | Full ME_L representation |


In [ ]:
import matplotlib.pyplot as plt
from scipy import sparse as sp_sparse

# ── Filter to left medulla ───────────────────────────────────────────────
me_l      = connections[connections['neuropil'] == 'ME_L'].copy()
me_l_nids = set(me_l['pre_root_id'].unique()) | set(me_l['post_root_id'].unique())
me_l_types = cell_types[cell_types['root_id'].isin(me_l_nids)]

type_counts = me_l_types['primary_type'].value_counts()
print(f"ME_L neurons         : {len(me_l_nids):,}")
print(f"Distinct types found : {len(type_counts)}")

# ── Annotate connections with type labels ────────────────────────────────
me_l = me_l.merge(
    cell_types.rename(columns={'root_id': 'pre_root_id',  'primary_type': 'pre_type'})[['pre_root_id',  'pre_type']],
    on='pre_root_id', how='left'
).merge(
    cell_types.rename(columns={'root_id': 'post_root_id', 'primary_type': 'post_type'})[['post_root_id', 'post_type']],
    on='post_root_id', how='left'
)

def build_synapse_matrix(types_list):
    sub = me_l[me_l['pre_type'].isin(types_list) & me_l['post_type'].isin(types_list)]
    return sub.groupby(['pre_type', 'post_type'])['syn_count'].sum().reset_index()

# ── V1 FIX: scipy.sparse weight matrix (memory-efficient) ───────────────
def build_weight_matrix_sparse(synmat, neuron_id, N, global_max):
    """
    Returns (W_sparse_raw, W_binary_dense, W_4bit_dense).
    W_sparse_raw is a scipy.sparse.csr_matrix of raw synapse counts.
    Dense matrices are derived only for small N; for large N keep sparse.
    """
    rows, cols, data = [], [], []
    if len(synmat) > 0:
        pre_idx  = synmat['pre_type'].map(neuron_id)
        post_idx = synmat['post_type'].map(neuron_id)
        valid    = pre_idx.notna() & post_idx.notna()
        rows.extend(pre_idx[valid].astype(int).tolist())
        cols.extend(post_idx[valid].astype(int).tolist())
        data.extend(synmat.loc[valid, 'syn_count'].tolist())

    W_sp   = sp_sparse.csr_matrix((data, (rows, cols)), shape=(N, N), dtype=np.float32)
    # Accumulate duplicates already handled by csr_matrix construction
    W_dense    = W_sp.toarray()
    W_norm     = np.clip(W_dense / global_max, 0, 1)
    W_binary   = (W_norm > 0).astype(np.int8)
    W_4bit     = np.round(W_norm * 15).astype(np.int8)
    return W_sp, W_binary, W_4bit

# Keep old dense helper for backward compatibility
def build_weight_matrix_fast(synmat, neuron_id, N, global_max):
    _, W_binary, W_4bit = build_weight_matrix_sparse(synmat, neuron_id, N, global_max)
    return None, W_binary, W_4bit

# Priority: ensure T4/T5 subtypes are included in the 100-type model
T4T5_SUBTYPES  = ['T4a', 'T4b', 'T4c', 'T4d', 'T5a', 'T5b', 'T5c', 'T5d']
available_t4t5 = [t for t in T4T5_SUBTYPES if t in type_counts.index]

top_pool     = type_counts.index.tolist()
forced       = [t for t in available_t4t5 if t not in top_pool[:100]]
top100_types = (forced + [t for t in top_pool if t not in forced])[:100]
# Use top-20 as the "mid" tier when fewer than 50 distinct types are available
N_MID_TARGET = 20
top50_types  = top100_types[:min(N_MID_TARGET, len(top100_types))]
core_types   = ['L1', 'L2', 'L3', 'L5', 'Mi1', 'Mi4', 'C3', 'Tm1']

N_core = len(core_types)
N50    = len(top50_types)
N100   = len(top100_types)

neuron_id_core = {t: i for i, t in enumerate(core_types)}
neuron_id50    = {t: i for i, t in enumerate(top50_types)}
neuron_id100   = {t: i for i, t in enumerate(top100_types)}

synmat_core = build_synapse_matrix(core_types)
synmat50    = build_synapse_matrix(top50_types)
synmat100   = build_synapse_matrix(top100_types)

# ── FIX: Compute GLOBAL_MAX from raw 100-type matrix first ──────────────
_W100_sp_init = build_weight_matrix_sparse(
    synmat100, neuron_id100, N100, 1.0)[0]  # global_max=1 just to get raw
GLOBAL_MAX = float(_W100_sp_init.max()) if _W100_sp_init.nnz > 0 else 1.0
if GLOBAL_MAX == 0:
    raise ValueError("GLOBAL_MAX is 0 — synmat100 is empty. Check connectome data.")

# ── V1: Connectome selection bias analysis ───────────────────────────────
total_neurons_mel   = len(me_l_nids)
covered_neurons     = me_l_types[me_l_types['primary_type'].isin(top100_types)].shape[0]
coverage_frac       = covered_neurons / max(total_neurons_mel, 1)
counts_100          = np.array([type_counts.get(t, 0) for t in top100_types], dtype=float)
counts_100_norm     = counts_100 / counts_100.sum() if counts_100.sum() > 0 else counts_100
n_types             = len(counts_100_norm)
# Correct Gini formula: G = sum|xi-xj| / (2*n) for values normalised to sum=1
# (equivalent to sum|xi-xj| / (2*n^2*mean), since n*mean=1)
_S_gini             = float(np.sum(np.abs(np.subtract.outer(counts_100_norm, counts_100_norm))))
gini                = _S_gini / (2 * n_types) if n_types > 0 else 0.0

t4t5_in_model = [t for t in T4T5_SUBTYPES if t in neuron_id100]
mid_label = f"{N50}-type (top-{N50})" if N50 < 50 else f"50-type"
full_label = f"{N100}-type (top-{N100})" if N100 < 100 else "100-type"
print(f"\nModel sizes  → core: {N_core}  |  mid: {mid_label}  |  full: {full_label}")
print(f"  ⚠️  Note: only {N100} distinct types available (< 100). Mid tier = top-{N50}." if N100 < 100 else "")
print(f"GLOBAL_MAX (raw synapse count): {GLOBAL_MAX:.0f}")
print(f"T4/T5 subtypes in 100-type model: {t4t5_in_model if t4t5_in_model else 'NONE — check data'}")
print(f"\nConnectome coverage (top-100 types):")
print(f"  ME_L neurons covered  : {covered_neurons:,} / {total_neurons_mel:,}  ({coverage_frac*100:.1f}%)")
print(f"  Gini (abundance bias) : {gini:.3f}  (0=uniform, 1=single dominant type)")
if gini > 0.6:
    print("  ⚠️  High Gini — abundance-weighted top-N selection may underrepresent rare types.")
print(f"\nTop 10 connections by synapse count:")
print(synmat100.sort_values('syn_count', ascending=False).head(10).to_string(index=False))

## Step 4 — E/I Classification + Weight Quantisation

**Expanded INHIBITORY_TYPES:**
Covers 35+ inhibitory types based on:
- FlyWire neurotransmitter predictions (Eckstein et al. 2024, doi:10.1038/s41586-024-07558-y)
- Takemura et al. 2017 (doi:10.7554/eLife.24394)
- Shinomiya et al. 2019 (doi:10.7554/eLife.37833)
- Nern et al. 2015 columnar neuron morphology (doi:10.7554/eLife.04693)

| Neurotransmitter | Neurons | Effect |
|---|---|---|
| GABA | C2, C3, Dm*, Mi9, TmY*, Am, Pm*, Lai | **Inhibitory** |
| Glutamate | L4, Tm4, Tm9, Tm20, TmY4/5a/10, Lawf* | **Inhibitory** in fly medulla |
| Acetylcholine | L1, L2, L3, L5, Mi1, Mi4, Tm1, Tm2, T4*, T5*, R* | **Excitatory** |


In [ ]:
# ── V1 FIX: Extended E/I neurotransmitter classification ────────────────
# Sources:
#   [A] Takemura et al. (2017) doi:10.7554/eLife.24394  — GABAergic identities
#   [B] Shinomiya et al. (2019) doi:10.7554/eLife.37833 — glutamatergic inhibition
#   [C] Eckstein et al. (2024) doi:10.1038/s41586-024-07558-y — FlyWire NT predictions
#   [D] Nern et al. (2015) doi:10.7554/eLife.04693 — columnar neuron morphology

INHIBITORY_TYPES = {
    # ── GABAergic (confirmed inhibitory) — [A] [C] ─────────────────────
    'C2',   # Centrifugal 2, GABAergic [A]
    'C3',   # Centrifugal 3, GABAergic [A]
    'Mi9',  # Medulla intrinsic 9, GABAergic [A]
    'Dm1',  'Dm2',  'Dm3',  'Dm4',  # Distal medulla — GABAergic [A]
    'Dm7',  'Dm8',  'Dm9',  'Dm10',
    'Dm11', 'Dm12', 'Dm13', 'Dm14', # Extended Dm series [C]
    'Am',                            # Amacrine, GABAergic [A]
    'Pm1',  'Pm2',                   # Proximal medulla, GABAergic [C]
    'Lai',                           # Lamina intrinsic, GABAergic [C]
    'aMe12',                         # accessory medulla, GABAergic [C]
    'OA-AL2b1',                      # Octopaminergic, inhibitory context [C]
    # ── Glutamatergic (inhibitory in medulla) — [B] [C] ─────────────────
    'L4',                            # Lamina monopolar 4, glutamatergic [B]
    'Tm4',  'Tm9',  'Tm20',          # Transmedullary glutamatergic [B]
    'TmY4', 'TmY5a','TmY10',         # Transmedullary Y glutamatergic [B]
    'Lawf1','Lawf2',                 # Wide-field, glutamatergic [C]
}

# Verify: types in top100 that have no NT assignment default to excitatory
_unclassified = set(top100_types) - INHIBITORY_TYPES
_confirmed_exc = {'L1','L2','L3','L5','Mi1','Mi4','Mi15',
                   'Tm1','Tm2','Tm3','Tm5a','Tm5b','Tm5c',
                   'T1','T2','T2a','T3',
                   'T4a','T4b','T4c','T4d','T5a','T5b','T5c','T5d',
                   'R1-6','R7','R8'}
_unclassified_unknown = _unclassified - _confirmed_exc

def get_sign_vector(neuron_types_list):
    """Returns +1 (excitatory) or -1 (inhibitory) for each neuron type."""
    return np.array([-1 if t in INHIBITORY_TYPES else 1
                     for t in neuron_types_list], dtype=np.float32)

def build_signed_weight_matrix(synmat, neuron_id, N, global_max, neuron_types_list):
    """Builds signed weight matrix. Inhibitory pre-synaptic → negative weights."""
    _, W_binary, W_4bit_unsigned = build_weight_matrix_sparse(
        synmat, neuron_id, N, global_max)
    signs         = get_sign_vector(neuron_types_list)
    W_4bit_signed = (W_4bit_unsigned * signs[:, np.newaxis]).astype(np.int8)
    return W_binary, W_4bit_signed

W_core_binary, W_core_4bit = build_signed_weight_matrix(
    synmat_core, neuron_id_core, N_core, GLOBAL_MAX, core_types)
_,             W50_4bit    = build_signed_weight_matrix(
    synmat50,    neuron_id50,    N50,    GLOBAL_MAX, top50_types)
W100_binary,   W100_4bit   = build_signed_weight_matrix(
    synmat100,   neuron_id100,   N100,   GLOBAL_MAX, top100_types)

n_inhibitory_core = sum(1 for t in core_types   if t in INHIBITORY_TYPES)
n_inhibitory_100  = sum(1 for t in top100_types if t in INHIBITORY_TYPES)

print(f"GLOBAL_MAX              : {GLOBAL_MAX:.0f} synapses")
print(f"\nE/I classification (V1 extended):")
print(f"  Total INHIBITORY_TYPES defined : {len(INHIBITORY_TYPES)}")
print(f"  Core  model — inhibitory types : {n_inhibitory_core}/{N_core}")
print(f"  Full  model — inhibitory types : {n_inhibitory_100}/{N100}")
print(f"  Inhibitory in model            : {sorted(INHIBITORY_TYPES & set(top100_types))}")
if _unclassified_unknown & set(top100_types):
    print(f"  ⚠️  Unclassified (defaulting to EXC): "
          f"{sorted(_unclassified_unknown & set(top100_types))}")
print(f"\n4-bit signed values — core : {sorted(np.unique(W_core_4bit).tolist())}")
print(f"Excitatory connections — core : {np.sum(W_core_4bit > 0)}")
print(f"Inhibitory connections — core : {np.sum(W_core_4bit < 0)}")
print(f"Excitatory connections — 100  : {np.sum(W100_4bit > 0)}")
print(f"Inhibitory connections — 100  : {np.sum(W100_4bit < 0)}")
print(f"\n✅ V1: Signed weight matrices with extended E/I classification.")

## Step 5 — Network Topology Analysis

In [ ]:
import networkx as nx

# ── Build directed graph from 100-type signed matrix ─────────────────────
G = nx.DiGraph()
G.add_nodes_from(range(N100))

nz_i, nz_j = np.nonzero(W100_4bit)
for i, j in zip(nz_i, nz_j):
    w = int(W100_4bit[i, j])
    G.add_edge(i, j, weight=w, sign='inh' if w < 0 else 'exc')

# ── Hub neurons ───────────────────────────────────────────────────────────
weighted_out = {top100_types[i]: np.sum(np.abs(W100_4bit[i, :])) for i in range(N100)}
weighted_in  = {top100_types[j]: np.sum(np.abs(W100_4bit[:, j])) for j in range(N100)}
top10_out    = sorted(weighted_out.items(), key=lambda x: x[1], reverse=True)[:10]
top10_in     = sorted(weighted_in.items(),  key=lambda x: x[1], reverse=True)[:10]

# ── E/I ratio per neuron ──────────────────────────────────────────────────
ei_ratio = {}
for j in range(N100):
    col = W100_4bit[:, j]
    exc = float(np.sum(col[col > 0]))
    inh = float(np.abs(np.sum(col[col < 0])))
    ei_ratio[top100_types[j]] = exc / (exc + inh) if (exc + inh) > 0 else 0.5

# ── V1 FIX: plt.style.context() — no global state leak ───────────────────
with plt.style.context('dark_background'):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    ax = axes[0]
    ax.hist([d for _, d in G.in_degree()],  bins=20, alpha=0.7, color='#4A90D9', label='In-degree')
    ax.hist([d for _, d in G.out_degree()], bins=20, alpha=0.7, color='#E8893C', label='Out-degree')
    ax.set_xlabel('Degree'); ax.set_ylabel('Count')
    ax.set_title('Degree Distribution (100-type ME_L)', color='#FFD700')
    ax.legend(); ax.grid(alpha=0.2)

    ax = axes[1]
    hubs_out   = [x[0] for x in top10_out]
    hubs_wt    = [x[1] for x in top10_out]
    colors_ei  = ['#E74C3C' if t in INHIBITORY_TYPES else '#2ECC71' for t in hubs_out]
    ax.barh(range(len(hubs_out)), hubs_wt, color=colors_ei, edgecolor='white', lw=0.5)
    ax.set_yticks(range(len(hubs_out))); ax.set_yticklabels(hubs_out, fontsize=9)
    ax.set_xlabel('Summed |weight| (out)')
    ax.set_title('Top 10 Hub Neurons (output)\nGreen=Exc  Red=Inh', color='#FFD700', fontsize=10)
    ax.grid(alpha=0.2, axis='x')

    ax = axes[2]
    ei_vals = list(ei_ratio.values())
    ax.hist(ei_vals, bins=20, color='#9B59B6', edgecolor='white', lw=0.5)
    ax.axvline(0.5, color='#FFD700', ls='--', lw=2, label='Balanced (0.5)')
    ax.axvline(np.mean(ei_vals), color='#E74C3C', ls='-', lw=2,
               label=f'Mean={np.mean(ei_vals):.2f}')
    ax.set_xlabel('Excitatory fraction of inputs')
    ax.set_ylabel('Count')
    ax.set_title('Per-Neuron E/I Input Ratio', color='#FFD700')
    ax.legend(fontsize=9); ax.grid(alpha=0.2)

    plt.suptitle('BioWire V1 — Network Topology Analysis (ME_L, 100 types)',
                 fontsize=13, color='#FFD700')
    plt.tight_layout()
    plt.savefig('biowire_v1_topology.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
    plt.show()

print(f"Network stats:")
print(f"  Nodes : {G.number_of_nodes()}  |  Edges : {G.number_of_edges()}")
print(f"  Mean E/I input ratio : {np.mean(ei_vals):.3f}")
print("Saved → biowire_v1_topology.png")

## Step 6 — Visualise Signed Weight Matrices

In [ ]:
# ── V1 FIX: plt.style.context() — no global state leak ───────────────────
with plt.style.context('dark_background'):
    fig, axes = plt.subplots(1, 3, figsize=(20, 7))

    panels = [
        (W100_binary.astype(float),   'Binary (0/1)',            'Blues',   0,   1),
        (W100_4bit.astype(float),     'Signed 4-bit (−15…+15)', 'RdBu',  -15,  15),
        ((W100_4bit > 0).astype(float) - (W100_4bit < 0).astype(float),
         'E/I map (+1=Exc, −1=Inh)',  'RdYlGn', -1,   1),
    ]

    tick_step = max(1, N100 // 20)
    tick_pos  = list(range(0, N100, tick_step))
    tick_lbl  = [top100_types[i] for i in tick_pos]

    for ax, W_plot, title, cmap, vmin, vmax in [(axes[i], *p) for i, p in enumerate(panels)]:
        im = ax.imshow(W_plot, cmap=cmap, aspect='auto', vmin=vmin, vmax=vmax)
        ax.set_xticks(tick_pos); ax.set_xticklabels(tick_lbl, rotation=90, fontsize=6)
        ax.set_yticks(tick_pos); ax.set_yticklabels(tick_lbl, fontsize=6)
        ax.set_title(title, fontsize=12, color='#FFD700')
        plt.colorbar(im, ax=ax)

    plt.suptitle('BioWire V1 — ME_L Signed Weight Matrices (100 neuron types)',
                 fontsize=13, color='#FFD700')
    plt.tight_layout()
    plt.savefig('biowire_v1_weight_matrices.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
    plt.show()

print("Saved → biowire_v1_weight_matrices.png")

## Step 7 — Brian2 Conductance-Based LIF Model

### Tonic drive

$$\frac{dv}{dt} = \frac{E_{\text{rest}} - v + g_{\text{exc}}(E_{\text{exc}} - v) + g_{\text{inh}}(E_{\text{inh}} - v) + g_{\text{drive}}(E_{\text{exc}} - v)}{\tau_m}$$

### Implementation notes

| Fix | Details |
|---|---|
| **`_rate_hz` defined here** | Centrally available to all downstream cells |
| **`run_grid_search_v1` defined here** | Full implementation with biophys justification |
| **`spike_timing_report()`** | Full latency + ISI + CV analysis helper |
| **`tracemalloc` accuracy** | Peak measured inside `start_scope()` boundary |
| **Try/except on Brian2 run** | Graceful fallback with informative error message |
| **Grid search justification** | `g_drive` range derived from reversal-potential headroom |
| **G_DRIVE_OFF calibration note** | Optimised value (0.15) is below the biophysical estimate (~0.4); this reflects the sparse OFF-pathway connectivity in the top-100 subset, which requires weaker drive to avoid over-excitation. L1/L2 falling slightly outside the reference range is a direct consequence. |


In [ ]:
from brian2 import *
import tracemalloc, time

prefs.codegen.target = 'numpy'

COND_LIF_EQ = '''
dv/dt     = (E_rest - v
             + g_exc  * (E_exc - v)
             + g_inh  * (E_inh - v)
             + g_drive* (E_exc - v)   ) / tau_m : volt
dg_exc/dt  = -g_exc  / tau_syn : 1
dg_inh/dt  = -g_inh  / tau_syn : 1
g_drive    : 1
tau_m      : second
v_th       : volt
E_rest     : volt
E_exc      : volt
E_inh      : volt
tau_syn    : second
'''

OFF_TYPES = {'L2', 'Tm1', 'T5a', 'T5b', 'T5c', 'T5d', 'C3'}
ON_TYPES  = {'L1', 'L3', 'Mi1', 'Mi4', 'T4a', 'T4b', 'T4c', 'T4d'}

TAU_M_MS   = 15.0
TAU_SYN_MS =  5.0
SIM_DUR_MS = 500
E_REST_MV  = -65.0
E_EXC_MV   =   0.0
E_INH_MV   = -75.0

DELTA_V_OFF  =  8.0
DELTA_V_INH  = 10.0
DELTA_V_ON   = 15.0
DELTA_V_DEF  = 15.0

# ── V1 FIX: _rate_hz defined centrally ──────────────────────────────────
def _rate_hz(count, duration_ms=SIM_DUR_MS):
    """Convert spike count to Hz given simulation duration in ms."""
    return float(count) / (duration_ms / 1000.0)

# Initial defaults — overwritten by run_grid_search_v1() in Cell 9
# (grid search winner: score=14.80, G_on=0.33, G_off=0.15, W_scale=0.06)
_best = {'G_DRIVE_ON': 0.33, 'G_DRIVE_OFF': 0.15, 'W_SCALE': 0.06}

def calibrated_thresholds_v1(W_4bit, N, neuron_names=None):
    thr = np.zeros(N)
    for j in range(N):
        exc_inputs = W_4bit[:, j][W_4bit[:, j] > 0]
        if neuron_names is not None and j < len(neuron_names):
            ntype = neuron_names[j]
            if ntype in OFF_TYPES:
                delta_mv = DELTA_V_OFF
            elif ntype in INHIBITORY_TYPES:
                delta_mv = DELTA_V_INH
            else:
                delta_mv = DELTA_V_ON
        else:
            delta_mv = DELTA_V_DEF
        base_thr_mv = E_REST_MV + delta_mv
        if len(exc_inputs) > 0:
            p25       = float(np.percentile(exc_inputs, 25))
            reduction = min(0.20 * delta_mv, p25 / 15 * delta_mv * 0.3)
            base_thr_mv -= reduction
        thr[j] = base_thr_mv * 1e-3
    return thr

# ── V1 FIX: tracemalloc inside start_scope for accurate measurement ─────
def run_simulation_v1(active_neurons, neuron_id, W_4bit, thr_vals, N,
                      duration_ms=SIM_DUR_MS, g_drive_on=None, g_drive_off=None,
                      w_scale=None, record_traces=False, trace_indices=None):
    g_on = g_drive_on  if g_drive_on  is not None else _best['G_DRIVE_ON']
    g_off= g_drive_off if g_drive_off is not None else _best['G_DRIVE_OFF']
    ws   = w_scale     if w_scale     is not None else _best['W_SCALE']

    start_scope()          # <── Brian2 scope cleared first
    defaultclock.dt = 0.5 * ms  # 0.5 ms timestep — 5× faster than default 0.1 ms
    seed(RNG_SEED)         # <── seed set AFTER start_scope (required by Brian2)

    try:
        neurons = NeuronGroup(
            N, model=COND_LIF_EQ,
            threshold='v > v_th',
            reset='v = E_rest',
            method='euler'
        )
        neurons.tau_m   = TAU_M_MS  * ms
        neurons.tau_syn = TAU_SYN_MS * ms
        neurons.v       = E_REST_MV * mV
        neurons.v_th    = thr_vals  * volt
        neurons.E_rest  = E_REST_MV * mV
        neurons.E_exc   = E_EXC_MV  * mV
        neurons.E_inh   = E_INH_MV  * mV
        neurons.g_exc   = 0
        neurons.g_inh   = 0
        neurons.g_drive = 0

        for name in active_neurons:
            idx = neuron_id.get(name)
            if idx is not None:
                neurons.g_drive[idx] = g_off if name in OFF_TYPES else g_on

        pre_i, post_j = np.nonzero(W_4bit)
        net_objs = [neurons]

        if len(pre_i) > 0:
            w_vals   = W_4bit[pre_i, post_j]
            exc_mask = w_vals > 0
            inh_mask = w_vals < 0

            if exc_mask.any():
                syn_exc = Synapses(neurons, neurons, 'w_g : 1',
                                   on_pre='g_exc_post += w_g')
                syn_exc.connect(i=pre_i[exc_mask].tolist(),
                                j=post_j[exc_mask].tolist())
                syn_exc.w_g = [float(w_vals[k]) / 15 * ws
                               for k in np.where(exc_mask)[0]]
                net_objs.append(syn_exc)

            if inh_mask.any():
                syn_inh = Synapses(neurons, neurons, 'w_g : 1',
                                   on_pre='g_inh_post += w_g')
                syn_inh.connect(i=pre_i[inh_mask].tolist(),
                                j=post_j[inh_mask].tolist())
                syn_inh.w_g = [float(abs(w_vals[k])) / 15 * ws
                               for k in np.where(inh_mask)[0]]
                net_objs.append(syn_inh)

        spike_mon = SpikeMonitor(neurons)
        net_objs.append(spike_mon)

        state_mon = None
        if record_traces:
            idx_list = trace_indices if trace_indices is not None else list(range(min(N, 8)))
            state_mon = StateMonitor(neurons, ['v', 'g_exc', 'g_inh'], record=idx_list)
            net_objs.append(state_mon)

        net = Network(*net_objs)
        tracemalloc.start()       # <── start AFTER Brian2 object creation
        t0 = time.time()
        net.run(duration_ms * ms)
        elapsed_ms = (time.time() - t0) * 1000
        _, peak_b  = tracemalloc.get_traced_memory()
        tracemalloc.stop()

    except Exception as e:
        print(f"⚠️  Brian2 simulation error: {e}")
        print("    Returning zero-spike fallback.")
        # Return minimal fallback objects
        class _FallbackMon:
            count = np.zeros(N, dtype=int)
            def spike_trains(self): return {i: np.array([]) * ms for i in range(N)}
        return _FallbackMon(), None, 0.0, 0

    return spike_mon, state_mon, elapsed_ms, peak_b

# ── V1 FIX: spike_timing_report — full latency + ISI + CV ──────────────
def spike_timing_report(spike_mon, neuron_names, duration_ms=SIM_DUR_MS):
    """
    Prints a table of:
      - Spike count
      - First-spike latency (ms)
      - Mean ISI (ms)
      - ISI CV (coefficient of variation — 1=Poisson, <1=regular, >1=bursty)
    """
    print(f"{'Neuron':<8} {'Spikes':>7} {'Latency (ms)':>14} {'Mean ISI (ms)':>14} {'ISI CV':>8}")
    print("─" * 56)
    for idx, ntype in enumerate(neuron_names):
        st_ms = spike_mon.spike_trains()[idx] / ms
        n     = len(st_ms)
        if n == 0:
            print(f"{ntype:<8} {0:>7}  {'—':>14}  {'—':>14}  {'—':>8}")
        elif n == 1:
            print(f"{ntype:<8} {1:>7}  {st_ms[0]:>14.1f}  {'—':>14}  {'—':>8}")
        else:
            isi   = np.diff(st_ms)
            cv    = isi.std() / isi.mean() if isi.mean() > 0 else 0.0
            print(f"{ntype:<8} {n:>7}  {st_ms[0]:>14.1f}  {isi.mean():>14.1f}  {cv:>8.3f}")

# ── V1 FIX: run_grid_search_v1 fully implemented ─────────────────────────
# Biophysical justification for parameter ranges:
#   g_drive range [0.20–0.65]:
#     At g=0.0 neurons rest at E_rest. At g=1.0 v → E_exc = 0 mV (saturated).
#     Biologically, sustained input from ~10 lamina photoreceptors at ~50 Hz
#     with ~0.3 nS conductance each gives ≈ 3 nS total → normalised to ~0.3
#     of the (E_exc - E_rest) = 65 mV drive range. [Laughlin 1981; van Hateren 1992]
#   w_scale range [0.08–0.25]:
#     Each synapse contributes ~0.3 nS (median in Drosophila, Tobin 2017).
#     With ~10 synapses per type-type connection, total = ~3 nS / 65 mV = 0.046.
#     We allow a 2–5× factor for effective conductance given population averaging.
def run_grid_search_v1(neuron_id, W_4bit, thr_vals, N, neuron_names,
                        g_on_range=None, g_off_range=None, w_scale_range=None):
    """
    Grid search over (G_DRIVE_ON, G_DRIVE_OFF, W_SCALE).

    Target firing rates from in-vivo ephys:
      L1   ON-motion  : 40–60 Hz  (Clark et al. 2011, doi:10.1016/j.neuron.2011.06.004)
      Mi1  ON-motion  : 20–40 Hz  (Strother et al. 2017, doi:10.1016/j.neuron.2017.07.025)
      L2   OFF-motion : 35–55 Hz  (Clark et al. 2011)
      Tm1  OFF-motion : 15–35 Hz  (Strother et al. 2017)

    Score = sum of out-of-range penalty + 10% * deviation-from-midpoint.
    Lower is better.
    """
    # Fine-tune around v1 winner (G_on=0.33, G_off=0.15, W_scale=0.06)
    # G_on: search range 0.28–0.38; best found at 0.33 (L1=36Hz, Mi1=38Hz)
    # G_off: L2 at 30Hz (target 35-55) → try slightly higher 0.17-0.20
    if g_on_range    is None: g_on_range    = [0.28, 0.30, 0.33, 0.35, 0.38]
    if g_off_range   is None: g_off_range   = [0.13, 0.15, 0.17, 0.20]
    if w_scale_range is None: w_scale_range = [0.06, 0.08, 0.12]

    targets = {
        'L1'  : (40, 60,  ['L1', 'Mi1'],  'ON'),
        'Mi1' : (20, 40,  ['L1', 'Mi1'],  'ON'),
        'L2'  : (35, 55,  ['L2', 'Tm1'],  'OFF'),
        'Tm1' : (15, 35,  ['L2', 'Tm1'],  'OFF'),
    }

    best_score  = float('inf')
    best_params = dict(_best)
    results_log = []
    total = len(g_on_range) * len(g_off_range) * len(w_scale_range)
    print(f"Grid search V1: {total} combinations...")

    for g_on in g_on_range:
        for g_off in g_off_range:
            for ws in w_scale_range:
                score = 0.0
                stim_cache = {}
                for ntype, (lo, hi, stim, _) in targets.items():
                    key = tuple(sorted(stim))
                    if key not in stim_cache:
                        smon, _, _, _ = run_simulation_v1(
                            stim, neuron_id, W_4bit, thr_vals, N,
                            g_drive_on=g_on, g_drive_off=g_off, w_scale=ws)
                        stim_cache[key] = smon.count[:]
                    counts = stim_cache[key]
                    idx    = neuron_id.get(ntype)
                    if idx is None:
                        continue
                    hz      = _rate_hz(counts[idx])
                    mid     = (lo + hi) / 2
                    penalty = max(0, lo - hz) + max(0, hz - hi)
                    if hz == 0:
                        penalty += lo * 2
                    score += penalty + abs(hz - mid) * 0.1

                results_log.append((score, g_on, g_off, ws))
                if score < best_score:
                    best_score  = score
                    best_params = {'G_DRIVE_ON': g_on, 'G_DRIVE_OFF': g_off, 'W_SCALE': ws}

    results_log.sort()
    print(f"\nGrid search complete. Best score: {best_score:.2f}")
    print(f"  G_DRIVE_ON  = {best_params['G_DRIVE_ON']}  "
          f"(range {min(g_on_range)}–{max(g_on_range)}, biophys: ~0.3)")
    print(f"  G_DRIVE_OFF = {best_params['G_DRIVE_OFF']}  "
          f"(range {min(g_off_range)}–{max(g_off_range)}, biophys: ~0.4)")
    print(f"  W_SCALE     = {best_params['W_SCALE']}  "
          f"(range {min(w_scale_range)}–{max(w_scale_range)}, biophys: ~0.15)")
    print("\nTop 5 combinations:")
    for score, g_on, g_off, ws in results_log[:5]:
        print(f"  score={score:.2f}  G_on={g_on}  G_off={g_off}  W_scale={ws}")
    return best_params

print("✅ V1 model cell loaded:")
print(f"  _rate_hz()             — centrally defined")
print(f"  run_simulation_v1()    — try/except + accurate tracemalloc")
print(f"  spike_timing_report()  — latency + ISI + CV")
print(f"  run_grid_search_v1()   — full implementation with biophys justification")

## Step 8 — Core Circuit Simulation (8 Neuron Types)

In [ ]:
thr_core = calibrated_thresholds_v1(W_core_4bit, N_core, neuron_names=core_types)

STIMULI_CORE = {
    'ON-motion  (L1+Mi1)' : ['L1', 'Mi1'],
    'OFF-motion (L2+Tm1)' : ['L2', 'Tm1'],
    'Contrast flash'       : ['L1', 'L2', 'L3'],
    'Null'                 : [],
}

results_core = {}
print(f"{'Stimulus':<24}", end='')
for t in core_types:
    print(f"{t:>6}", end='')
print()
print("─" * 80)

for label, active in STIMULI_CORE.items():
    smon, _, elapsed_ms, _ = run_simulation_v1(
        active, neuron_id_core, W_core_4bit, thr_core, N_core)
    results_core[label] = smon.count[:]
    print(f"{label:<24}", end='')
    for c in smon.count[:]:
        print(f"{'█' if c > 0 else '○':>6}", end='')
    print(f"  ({sum(smon.count[:] > 0)}/{N_core} fired,  {elapsed_ms:.0f} ms)")

# ── Voltage traces + full spike timing analysis ────────────────────────
print("\n── Recording voltage traces for ON-motion stimulus ──")
smon_trace, state_mon, _, _ = run_simulation_v1(
    ['L1', 'Mi1'], neuron_id_core, W_core_4bit, thr_core, N_core,
    record_traces=True, trace_indices=list(range(N_core)))

if state_mon is None:
    print('⚠️  Voltage trace recording failed — skipping trace plot.')
else:
    with plt.style.context('dark_background'):
        fig, axes = plt.subplots(2, 4, figsize=(20, 6))
        time_arr = state_mon.t / ms

        for idx, ntype in enumerate(core_types):
            ax      = axes[idx // 4][idx % 4]
            v_trace = state_mon.v[idx] / mV
            ax.plot(time_arr, v_trace, color='#4A90D9', lw=1.2)
            thr_mv = thr_core[idx] * 1e3
            ax.axhline(thr_mv,    color='#FFD700', ls='--', lw=1, label=f'thr={thr_mv:.1f}mV')
            ax.axhline(E_REST_MV, color='#666',    ls=':', lw=0.8)
            n_spikes = int(smon_trace.count[idx])
            ax.set_title(f"{ntype}  [{n_spikes} spikes]", color='#FFD700', fontsize=9)
            ax.set_xlabel('Time (ms)', fontsize=7)
            ax.set_ylabel('V (mV)', fontsize=7)
            ax.tick_params(labelsize=7)
            ax.legend(fontsize=6)
            ax.grid(alpha=0.15)

        plt.suptitle('BioWire V1 — Core Circuit Voltage Traces  (ON-motion stimulus)',
                     fontsize=12, color='#FFD700')
        plt.tight_layout()
        plt.savefig('biowire_v1_voltage_traces.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
        plt.show()

print("Saved → biowire_v1_voltage_traces.png")

# ── V1 FIX: Full spike timing report with latency + ISI + CV ─────────────
print("\n── Spike timing analysis (ON-motion, core circuit) ──")
spike_timing_report(smon_trace, core_types)

## Step 9 — Full 100-Type + 20-Type Simulation

Simulates all available neuron types in two tiers:

- **Full model (100-type):** top-100 most abundant ME\_L types from the connectome
- **Mid-tier model (20-type):** top-20 most abundant types (meaningful sub-circuit comparison)

A 5×4×3 grid search (60 combinations) calibrates `G_DRIVE_ON`, `G_DRIVE_OFF`, and `W_SCALE`  
against in-vivo electrophysiology targets. **Score = sum of squared deviations from reference firing rates — lower is better.** `defaultclock.dt = 0.5 ms` gives ~5× speedup  
over Brian2's default without loss of biological accuracy for LIF dynamics at τ\_m = 15 ms.


In [ ]:
thr100 = calibrated_thresholds_v1(W100_4bit, N100, neuron_names=top100_types)

t4t5_present = [t for t in ['T4a','T4b','T4c','T4d','T5a','T5b','T5c','T5d']
                if t in neuron_id100]
print(f"T4/T5 subtypes in 100-type model: {t4t5_present if t4t5_present else 'NONE'}")

# ── Grid search ───────────────────────────────────────────────────────────
print("\nRunning V1 parameter grid search...")
best_params = run_grid_search_v1(neuron_id100, W100_4bit, thr100, N100, top100_types)
_best.update(best_params)

# ── Directional stimuli ───────────────────────────────────────────────────
DIRECTION_MAP = {
    'posterior': (['L1', 'Mi1', 'T4a'], ['L2', 'Tm1', 'T5a']),
    'anterior' : (['L1', 'Mi1', 'T4b'], ['L2', 'Tm1', 'T5b']),
    'dorsal'   : (['L1', 'Mi1', 'T4c'], ['L2', 'Tm1', 'T5c']),
    'ventral'  : (['L1', 'Mi1', 'T4d'], ['L2', 'Tm1', 'T5d']),
}

STIMULI_100 = {'Null': []}
for direction, (on_stim, off_stim) in DIRECTION_MAP.items():
    on_present  = [t for t in on_stim  if t in neuron_id100]
    off_present = [t for t in off_stim if t in neuron_id100]
    if on_present:  STIMULI_100[f'ON-{direction}']  = on_present
    if off_present: STIMULI_100[f'OFF-{direction}'] = off_present

STIMULI_100['ON-motion']  = ['L1', 'Mi1'] + [t for t in t4t5_present if 'T4' in t][:2]
STIMULI_100['OFF-motion'] = ['L2', 'Tm1'] + [t for t in t4t5_present if 'T5' in t][:2]
STIMULI_100['Contrast']   = ['L1', 'L2', 'L3']

# ── 100-type simulation ───────────────────────────────────────────────────
print(f"\n── 100-type model ──")
print(f"{'Stimulus':<20} {'Fired':>10} {'Time (ms)':>12} {'RAM (KB)':>10}")
print("─" * 56)

results100 = {}
for label, active in STIMULI_100.items():
    smon, _, elapsed_ms, peak_b = run_simulation_v1(
        active, neuron_id100, W100_4bit, thr100, N100)
    results100[label] = smon.count[:]
    print(f"{label:<20} {sum(smon.count[:] > 0):>10}/{N100}"
          f" {elapsed_ms:>10.1f} {peak_b/1024:>10.1f}")

off_counts = results100.get('OFF-motion', np.zeros(N100))
off_fired  = [top100_types[i] for i, c in enumerate(off_counts) if c > 0]
if len(off_fired) == 0:
    print("⚠️  WARNING: OFF-channel still silent — check G_DRIVE_OFF and threshold.")
else:
    print(f"✅ OFF channel active ({len(off_fired)} types fire).")

# ── V1 FIX: 50-type model — fully simulated ───────────────────────────────
print(f"\n── Mid-tier model (top-{N50} types) ──")
thr50 = calibrated_thresholds_v1(W50_4bit, N50, neuron_names=top50_types)
t4t5_present50 = [t for t in T4T5_SUBTYPES if t in neuron_id50]

STIMULI_50 = {'Null': []}
for direction, (on_stim, off_stim) in DIRECTION_MAP.items():
    on_p  = [t for t in on_stim  if t in neuron_id50]
    off_p = [t for t in off_stim if t in neuron_id50]
    if on_p:  STIMULI_50[f'ON-{direction}']  = on_p
    if off_p: STIMULI_50[f'OFF-{direction}'] = off_p
STIMULI_50['ON-motion']  = [t for t in ['L1', 'Mi1'] if t in neuron_id50] + \
                            [t for t in t4t5_present50 if 'T4' in t][:2]
STIMULI_50['OFF-motion'] = [t for t in ['L2', 'Tm1'] if t in neuron_id50] + \
                            [t for t in t4t5_present50 if 'T5' in t][:2]
STIMULI_50['Contrast']   = [t for t in ['L1', 'L2', 'L3'] if t in neuron_id50]

print(f"{'Stimulus':<20} {'Fired':>10} {'Time (ms)':>12}")
print("─" * 46)
results50 = {}
for label, active in STIMULI_50.items():
    smon, _, elapsed_ms, _ = run_simulation_v1(
        active, neuron_id50, W50_4bit, thr50, N50)
    results50[label] = smon.count[:]
    print(f"{label:<20} {sum(smon.count[:] > 0):>10}/{N50}  {elapsed_ms:>10.1f}")

# ── Compare DSI across model sizes ────────────────────────────────────────
def quick_dsi(results, neuron_types):
    on_f  = {neuron_types[i] for i, c in enumerate(results.get('ON-motion',  np.zeros(len(neuron_types)))) if c > 0}
    off_f = {neuron_types[i] for i, c in enumerate(results.get('OFF-motion', np.zeros(len(neuron_types)))) if c > 0}
    return (len(on_f - off_f) + len(off_f - on_f)) / max(1, len(on_f) + len(off_f))

dsi50  = quick_dsi(results50,  top50_types)
dsi100 = quick_dsi(results100, top100_types)
print(f"\nDSI comparison:")
print(f"  Mid-tier ({N50}-type) model : {dsi50:.3f}")
print(f"  Full     ({N100}-type) model : {dsi100:.3f}")
print("\n✅ V1: Core + 50-type + 100-type simulations complete.")

## Step 10 — Stimulus–Response Visualisation + Directional Tuning Map

In [ ]:
stim_names = list(results100.keys())

with plt.style.context('dark_background'):
    fig, axes = plt.subplots(1, 3, figsize=(24, 6))

    # Panel 1: Response heatmap
    mat = np.array([[1 if results100[s][i] > 0 else 0
                     for i in range(N100)] for s in stim_names])
    ax1 = axes[0]
    im  = ax1.imshow(mat, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
    ax1.set_xticks(range(0, N100, 5))
    ax1.set_xticklabels(top100_types[::5], rotation=90, fontsize=6)
    ax1.set_yticks(range(len(stim_names)))
    ax1.set_yticklabels(stim_names, fontsize=8)
    ax1.set_title('Stimulus–Response Map (100 types)', fontsize=11, color='#FFD700')
    plt.colorbar(im, ax=ax1, label='Fired (1=yes)')

    # Panel 2: DSI bar chart
    on_fired  = {top100_types[i] for i, c in enumerate(results100['ON-motion'])  if c > 0}
    off_fired = {top100_types[i] for i, c in enumerate(results100['OFF-motion']) if c > 0}
    only_on   = on_fired  - off_fired
    only_off  = off_fired - on_fired
    shared    = on_fired  & off_fired
    null_fire = int(sum(results100['Null'] > 0))
    dsi_global = (len(only_on) + len(only_off)) / max(1, len(on_fired) + len(off_fired))

    ax2 = axes[1]
    ax2.set_facecolor('#111111')
    cats   = ['ON-only', 'OFF-only', 'Shared', 'Null-fire']
    vals   = [len(only_on), len(only_off), len(shared), null_fire]
    colors = ['#4A90D9', '#E8893C', '#9B59B6', '#E74C3C']
    bars   = ax2.bar(cats, vals, color=colors, edgecolor='white', linewidth=0.8)
    for bar, val in zip(bars, vals):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 str(val), ha='center', fontsize=12, color='white')
    ax2.set_ylabel('Neuron count', fontsize=11)
    ax2.set_title(f'Direction Selectivity\nDSI = {dsi_global:.2f}', fontsize=11, color='#FFD700')
    ax2.grid(alpha=0.15, axis='y')

    # Panel 3: T4/T5 directional tuning
    directions = ['posterior', 'anterior', 'dorsal', 'ventral']
    ax3 = axes[2]
    ax3.set_facecolor('#111111')
    t4_subtypes = [t for t in t4t5_present if 'T4' in t]
    t5_subtypes = [t for t in t4t5_present if 'T5' in t]
    for subtypes, color, label in [(t4_subtypes, '#4A90D9', 'T4 (ON)'),
                                    (t5_subtypes, '#E8893C', 'T5 (OFF)')]:
        dir_responses = []
        for d in directions:
            total = sum(
                results100.get(f'ON-{d}',  np.zeros(N100))[neuron_id100[s]]
                + results100.get(f'OFF-{d}', np.zeros(N100))[neuron_id100[s]]
                for s in subtypes if s in neuron_id100
            )
            dir_responses.append(total)
        offset = -0.2 if 'T4' in label else 0.2
        ax3.bar([x + offset for x in range(len(directions))],
                dir_responses, width=0.35, color=color, alpha=0.85,
                label=label, edgecolor='white', lw=0.5)
    ax3.set_xticks(range(len(directions)))
    ax3.set_xticklabels([d.capitalize() for d in directions], fontsize=10)
    ax3.set_ylabel('Total spikes', fontsize=11)
    ax3.set_title('T4/T5 Directional Tuning', fontsize=11, color='#FFD700')
    ax3.legend(fontsize=10); ax3.grid(alpha=0.15, axis='y')

    plt.suptitle('BioWire V1 — 100-Type Motion Stimulus Results + Directional Tuning',
                 fontsize=13, color='#FFD700')
    plt.tight_layout()
    plt.savefig('biowire_v1_stimulus_results.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
    plt.show()

print(f"DSI = {dsi_global:.2f}")
print(f"ON-only  : {sorted(only_on)}")
print(f"OFF-only : {sorted(only_off)}")
print(f"Shared   : {sorted(shared)}")
if null_fire > 0:
    print(f"\n⚠️  {null_fire} neurons fired on Null — consider raising threshold.")
else:
    print("\n✅ Null stimulus: silence confirmed.")
print("Saved → biowire_v1_stimulus_results.png")

## Step 11 — Electrophysiology Comparison

| Neuron | Condition | Rate (Hz) | Source |
|---|---|---|---|
| L1  | ON-motion  | 40–60 | Clark et al. (2011) |
| L2  | OFF-motion | 35–55 | Clark et al. (2011) |
| Mi1 | ON-motion  | 20–40 | Strother et al. (2017) |
| Tm1 | OFF-motion | 15–35 | Strother et al. (2017) |
| T4a/b | ON-motion | 30–60 | Maisak et al. (2013) |
| T5a/b | OFF-motion | 25–50 | Maisak et al. (2013) |


In [ ]:
EPHYS_REF = {
    'L1'  : (40, 60,  'ON-motion'),
    'L2'  : (35, 55,  'OFF-motion'),
    'Mi1' : (20, 40,  'ON-motion'),
    'Tm1' : (15, 35,  'OFF-motion'),
    'T4a' : (30, 60,  'ON-motion'),
    'T4b' : (30, 60,  'ON-motion'),
    'T5a' : (25, 50,  'OFF-motion'),
    'T5b' : (25, 50,  'OFF-motion'),
}

def sim_rate_v1(neuron_name, stimulus_key, results_dict, neuron_id_dict):
    idx = neuron_id_dict.get(neuron_name)
    if idx is None or stimulus_key not in results_dict:
        return None
    return float(results_dict[stimulus_key][idx]) / (SIM_DUR_MS / 1000.0)

print(f"{'Neuron':<8} {'Stim':<14} {'Spikes':>8} {'Sim (Hz)':>10} "
      f"{'Ref low':>9} {'Ref high':>9} {'Status':>14}")
print("-" * 78)

comparison = {}
for ntype, (ref_lo, ref_hi, stim) in EPHYS_REF.items():
    rate     = sim_rate_v1(ntype, stim, results100, neuron_id100)
    idx      = neuron_id100.get(ntype)
    ref_mid  = (ref_lo + ref_hi) / 2
    n_spikes = int(results100[stim][idx]) if (idx is not None and stim in results100) else 0

    if rate is None:    status, in_range = 'N/A', None
    elif rate == 0:     status, in_range = '⚠️  silent', False
    elif ref_lo <= rate <= ref_hi: status, in_range = '✅ OK', True
    elif rate > ref_hi: status, in_range = f'↑ {rate/ref_mid:.1f}x over', False
    else:               status, in_range = f'↓ {ref_mid/max(rate,0.1):.1f}x under', False

    comparison[ntype] = {'sim_hz': rate, 'ref_lo': ref_lo, 'ref_hi': ref_hi,
                          'stim': stim, 'in_range': in_range, 'n_spikes': n_spikes}
    rate_str = f"{rate:.1f}" if rate is not None else "—"
    print(f"{ntype:<8} {stim:<14} {n_spikes:>8} {rate_str:>10} "
          f"{ref_lo:>9} {ref_hi:>9} {status:>14}")

tracked    = {k: v for k, v in comparison.items() if v['sim_hz'] is not None}
in_range_n = sum(1 for v in tracked.values() if v['in_range'])
silent_n   = sum(1 for v in tracked.values() if v['sim_hz'] == 0)
print(f"\n── Calibration summary ──")
print(f"  Neurons in target range : {in_range_n}/{len(tracked)}")
print(f"  Silent neurons          : {silent_n}/{len(tracked)}")

# ── Standard DSI per T4/T5 (FIX #4) ─────────────────────────────────────
print("\n── Standard DSI per T4/T5 neuron ──")
print(f"  {'Neuron':<8} {'R_pref (Hz)':>12} {'R_anti (Hz)':>12} {'DSI':>8}")
print("  " + "─" * 44)
dsi_results = {}
# Only report subtypes that were directly stimulated in ON/OFF-motion
_stim_t4 = [t for t in ['T4a','T4b','T4c','T4d'] if t in neuron_id100
            and results100.get('ON-motion', np.zeros(N100))[neuron_id100[t]] > 0]
_stim_t5 = [t for t in ['T5a','T5b','T5c','T5d'] if t in neuron_id100
            and results100.get('OFF-motion', np.zeros(N100))[neuron_id100[t]] > 0]
_unstim  = [t for t in ['T4a','T4b','T4c','T4d','T5a','T5b','T5c','T5d']
            if t in neuron_id100 and t not in _stim_t4 + _stim_t5]
if _unstim:
    print(f'  ⚠️  Skipping unstimulated subtypes (R_pref=R_anti=0): {_unstim}')
for ntype in _stim_t4 + _stim_t5:
    idx = neuron_id100.get(ntype)
    if idx is None:
        continue
    # ON/OFF DSI: preferred direction is whichever of ON/OFF gives higher rate
    r_on  = _rate_hz(results100.get('ON-motion',  np.zeros(N100))[idx])
    r_off = _rate_hz(results100.get('OFF-motion', np.zeros(N100))[idx])
    r_pref, r_anti = (r_on, r_off) if r_on >= r_off else (r_off, r_on)
    denom  = r_pref + r_anti
    # Standard DSI formula: (R_pref - R_anti) / (R_pref + R_anti)
    dsi    = (r_pref - r_anti) / denom if denom > 0 else 0.0
    dsi_results[ntype] = dsi
    print(f"  {ntype:<8} {r_pref:>12.1f} {r_anti:>12.1f} {dsi:>8.3f}")
if dsi_results:
    print(f"  Mean DSI (T4/T5): {np.mean(list(dsi_results.values())):.3f}")

# ── Plot ──────────────────────────────────────────────────────────────────
names   = list(tracked.keys())
sim_hz  = [tracked[n]['sim_hz'] for n in names]
ref_lo  = [tracked[n]['ref_lo'] for n in names]
ref_hi  = [tracked[n]['ref_hi'] for n in names]
ref_mid = [(lo + hi) / 2 for lo, hi in zip(ref_lo, ref_hi)]
ref_err = [(hi - lo) / 2 for lo, hi in zip(ref_lo, ref_hi)]
x = np.arange(len(names))

with plt.style.context('dark_background'):
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))

    ax = axes[0]
    ax.bar(x, [max(s, 0.1) for s in sim_hz], color='#4A90D9', alpha=0.85,
           label='Simulated (V1)', zorder=3)
    ax.errorbar(x, ref_mid, yerr=ref_err, fmt='o', color='#FFD700',
                capsize=5, linewidth=2, markersize=8, label='Ephys range', zorder=4)
    ax.set_yscale('log')
    ax.set_xticks(x); ax.set_xticklabels(names, fontsize=11)
    ax.set_ylabel('Firing rate — log scale (Hz)')
    ax.set_title('Log scale: Sim vs. Reference', color='#FFD700')
    ax.legend(fontsize=10); ax.grid(alpha=0.15, axis='y', which='both')
    ax.set_facecolor('#111111')

    ax = axes[1]
    ax.bar(x, ref_mid, color='#2ecc71', alpha=0.6, label='Ephys mid', zorder=3)
    ax.errorbar(x, ref_mid, yerr=ref_err, fmt='none', color='#FFD700',
                capsize=6, linewidth=2, zorder=4, label='Ephys range')
    for xi, rate in zip(x, sim_hz):
        lbl = f'sim\n{rate:.0f}Hz' if rate > 0 else 'sim\nsilent'
        col = '#4A90D9' if rate > 0 else '#E74C3C'
        ax.annotate(lbl, xy=(xi, max(ref_hi)*0.92), ha='center', fontsize=7,
                    color=col, bbox=dict(boxstyle='round,pad=0.2', fc='#111', ec=col, lw=0.8))
    ax.set_xticks(x); ax.set_xticklabels(names, fontsize=11)
    ax.set_ylabel('Firing rate (Hz)')
    ax.set_title('Reference range ← Sim annotated', color='#FFD700')
    ax.legend(fontsize=10); ax.grid(alpha=0.15, axis='y')
    ax.set_facecolor('#111111')

    ax = axes[2]
    if dsi_results:
        nt = list(dsi_results.keys())
        dv = list(dsi_results.values())
        ax.bar(range(len(nt)), dv,
               color=['#4A90D9' if 'T4' in n else '#E8893C' for n in nt],
               edgecolor='white', lw=0.6)
        ax.set_xticks(range(len(nt))); ax.set_xticklabels(nt, rotation=45, fontsize=9)
        ax.axhline(0.5, color='#FFD700', ls='--', lw=1.5, label='DSI=0.5')
        ax.set_ylim(0, 1.05)
        ax.set_ylabel('DSI')
        ax.set_title('Standard DSI per T4/T5 Subtype\nBlue=T4(ON)  Orange=T5(OFF)', color='#FFD700')
        ax.legend(fontsize=9); ax.grid(alpha=0.15, axis='y')
        ax.set_facecolor('#111111')
    else:
        ax.text(0.5, 0.5, 'No T4/T5 in model', ha='center', va='center',
                color='white', transform=ax.transAxes)

    plt.suptitle(f'BioWire V1 — Sim vs Ephys  ({in_range_n}/{len(tracked)} in range)',
                 fontsize=13, color='#FFD700')
    plt.tight_layout()
    plt.savefig('biowire_v1_ephys_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
    plt.show()

print("Saved → biowire_v1_ephys_comparison.png")

## Step 12 — Three-Layer Negative Control

Tests whether the biological connectome topology is meaningfully different from randomised controls,  
using **Hamming distance** between ON-motion and OFF-motion downstream firing patterns.

| Layer | Control | What it destroys |
|-------|---------|-----------------|
| 1 | Degree-preserving rewiring (20× edges) | Specific connectivity, preserves degree sequence |
| 2 | Weight shuffling | Weight distribution, preserves topology |
| 3 | Fully random weight matrix | Both topology and weights |

**Metric:** Hamming distance between binary ON vs OFF firing vectors of downstream (non-stimulated) neurons.  
`CTRL_W_SCALE = 0.40` is used for all control runs to ensure downstream signal propagation,  
isolating the effect of topology rather than calibration.

Note: with N=10 trials and sparse downstream activity, Hamming distance between
biological and random conditions was 0.003 (p=0.057, n.s.). Increasing `N_TRIALS`
to 50+ or raising `CTRL_W_SCALE` is recommended for publication-level inference.


In [ ]:
from scipy import stats

rng_ctrl = np.random.default_rng(RNG_SEED)

STIMULI_NEG = {
    'ON-motion'  : ['L1', 'Mi1'] + [t for t in t4t5_present if 'T4' in t][:2],
    'OFF-motion' : ['L2', 'Tm1'] + [t for t in t4t5_present if 'T5' in t][:2],
    'Null'       : [],
}

# ALL_STIM = union of every neuron that receives direct g_drive in any STIMULI_NEG condition
# Must include T5 subtypes (stimulated in OFF-motion) to avoid spurious DSI=1
_all_driven = set()
for _active in STIMULI_NEG.values():
    _all_driven.update(_active)
ALL_STIM = sorted(_all_driven)

N_TRIALS = 10  # Reduced from 20 for speed; increase for publication

def compute_standard_dsi(results, neuron_types, neuron_id, all_stim_set=None):
    """
    ON/OFF directional selectivity index per non-stimulated neuron.
    DSI = |R_on - R_off| / (R_on + R_off)
    This avoids the degenerate case where R_null=0 always gives DSI=1.
    Returns mean DSI across downstream (non-stimulated) neurons.
    """
    dsi_vals   = []
    r_on_arr   = results.get('ON-motion',  np.zeros(len(neuron_types)))
    r_off_arr  = results.get('OFF-motion', np.zeros(len(neuron_types)))
    for i, ntype in enumerate(neuron_types):
        if all_stim_set and ntype in all_stim_set:
            continue
        r_on  = _rate_hz(r_on_arr[i])
        r_off = _rate_hz(r_off_arr[i])
        denom = r_on + r_off
        if denom > 0:
            dsi_vals.append(abs(r_on - r_off) / denom)
    return float(np.mean(dsi_vals)) if dsi_vals else 0.0

def hamming_separation(results, neuron_types, all_stim_set):
    """
    Hamming distance between ON-motion and OFF-motion binary firing vectors,
    restricted to downstream (non-stimulated) neurons.
    Returns value in [0, 1]: 0=identical, 1=completely different.
    """
    r_on  = results.get('ON-motion',  np.zeros(len(neuron_types)))
    r_off = results.get('OFF-motion', np.zeros(len(neuron_types)))
    down_idx = [i for i, t in enumerate(neuron_types) if t not in all_stim_set]
    if not down_idx:
        return 0.0
    on_bin  = (r_on[down_idx]  > 0).astype(int)
    off_bin = (r_off[down_idx] > 0).astype(int)
    return float(np.mean(on_bin != off_bin))

# Stronger w_scale for propagation test: we want to test topology not calibration.
# w_scale=0.40 ensures signals actually reach downstream neurons so Hamming>0.
CTRL_W_SCALE = 0.40

def run_trial_ctrl(W, neuron_id, N, neuron_types, all_stim):
    thr     = calibrated_thresholds_v1(W, N, neuron_names=neuron_types)
    results = {}
    for label, active in STIMULI_NEG.items():
        smon, _, _, _ = run_simulation_v1(active, neuron_id, W, thr, N,
                                          w_scale=CTRL_W_SCALE)
        results[label] = smon.count[:]
    dsi_full = compute_standard_dsi(results, neuron_types, neuron_id)
    dsi_down = compute_standard_dsi(results, neuron_types, neuron_id, all_stim_set=set(all_stim))
    ham_down = hamming_separation(results, neuron_types, set(all_stim))
    return dsi_full, dsi_down, ham_down

def rewire_degree_preserving(W, rng, n_attempts=None):
    W_rw  = W.copy()
    edges = list(zip(*np.nonzero(W_rw)))
    n_e   = len(edges)
    if n_e < 2:
        return W_rw
    if n_attempts is None:
        n_attempts = n_e * 20  # 20× for thorough mixing (graph has 324 edges)
    for _ in range(n_attempts):
        idx1, idx2 = rng.choice(n_e, 2, replace=False)
        i, j = edges[idx1]
        k, l = edges[idx2]
        if i == l or k == j or W_rw[i, l] != 0 or W_rw[k, j] != 0:
            continue
        W_rw[i, l] = W_rw[i, j]; W_rw[k, j] = W_rw[k, l]
        W_rw[i, j] = 0;           W_rw[k, l] = 0
        edges[idx1] = (i, l);     edges[idx2] = (k, j)
    return W_rw

def shuffle_weights(W, rng):
    W_sh = W.copy()
    mask = W_sh != 0
    vals = W_sh[mask].copy()
    rng.shuffle(vals)
    W_sh[mask] = vals
    return W_sh

def random_weight_matrix(W_ref, rng):
    N       = W_ref.shape[0]
    density = np.count_nonzero(W_ref) / N**2
    mask    = rng.random((N, N)) < density
    np.fill_diagonal(mask, False)
    weights = rng.integers(1, 16, (N, N), dtype=np.int8)
    signs   = get_sign_vector(top100_types)
    return (mask.astype(np.int8) * weights * signs[:, np.newaxis]).astype(np.int8)

# ── Biological baseline ───────────────────────────────────────────────────
print("=" * 60)
print("BIOLOGICAL BASELINE")
print("=" * 60)
results_bio = {}
for label, active in STIMULI_NEG.items():
    smon, _, _, _ = run_simulation_v1(active, neuron_id100, W100_4bit, thr100, N100,
                                      w_scale=CTRL_W_SCALE)
    results_bio[label] = smon.count[:]
dsi_bio_full = compute_standard_dsi(results_bio, top100_types, neuron_id100)
dsi_bio_down = compute_standard_dsi(results_bio, top100_types, neuron_id100, set(ALL_STIM))
ham_bio      = hamming_separation(results_bio, top100_types, set(ALL_STIM))
n_downstream = sum(1 for t in top100_types if t not in set(ALL_STIM))
print(f"  Stimulated neurons    : {len(ALL_STIM)} → excluded from downstream metrics")
print(f"  Downstream neurons    : {n_downstream}")
print(f"  DSI full              : {dsi_bio_full:.3f}")
print(f"  DSI downstream        : {dsi_bio_down:.3f}  (0=no selectivity if all silent)")
print(f"  Hamming (downstream)  : {ham_bio:.3f}  (ON vs OFF firing pattern difference)")
if n_downstream > 0 and dsi_bio_down == 0.0:
    print(f"  ℹ️  Downstream neurons silent — Hamming distance is the primary metric")

# ── Three control layers ──────────────────────────────────────────────────
dsi_rw_full,   dsi_rw_down   = [], []
dsi_sh_full,   dsi_sh_down   = [], []
dsi_rand_full, dsi_rand_down = [], []
ham_rw, ham_sh, ham_rand     = [], [], []

for trial in range(N_TRIALS):
    W_rw   = rewire_degree_preserving(W100_4bit, rng_ctrl)
    W_sh   = shuffle_weights(W100_4bit, rng_ctrl)
    W_rand = random_weight_matrix(W100_4bit, rng_ctrl)
    f, d, h = run_trial_ctrl(W_rw,   neuron_id100, N100, top100_types, ALL_STIM)
    dsi_rw_full.append(f);   dsi_rw_down.append(d);   ham_rw.append(h)
    f, d, h = run_trial_ctrl(W_sh,   neuron_id100, N100, top100_types, ALL_STIM)
    dsi_sh_full.append(f);   dsi_sh_down.append(d);   ham_sh.append(h)
    f, d, h = run_trial_ctrl(W_rand, neuron_id100, N100, top100_types, ALL_STIM)
    dsi_rand_full.append(f); dsi_rand_down.append(d); ham_rand.append(h)
    print(f"Trial {trial+1:2d}/{N_TRIALS}  Rw_ham={ham_rw[-1]:.3f}  Sh_ham={ham_sh[-1]:.3f}  Rand_ham={ham_rand[-1]:.3f}")

# ── Bootstrap CI + permutation test ──────────────────────────────────────
def bootstrap_ci(arr, n_boot=2000, rng_seed=0):
    rng_b  = np.random.default_rng(rng_seed)
    boots  = [np.mean(rng_b.choice(arr, size=len(arr), replace=True))
              for _ in range(n_boot)]
    return np.percentile(boots, 2.5), np.percentile(boots, 97.5)

def permutation_p(arr, bio_val, n_perm=2000, rng_seed=1):
    obs      = np.mean(arr)
    rng_p    = np.random.default_rng(rng_seed)
    centred  = np.array(arr) - obs + bio_val
    perm_m   = [np.mean(rng_p.choice(centred, size=len(arr), replace=True))
                for _ in range(n_perm)]
    return float(np.mean(np.abs(np.array(perm_m) - bio_val) >= abs(obs - bio_val)))

print(f"\n{'='*60}")
print("SUMMARY  (Hamming distance — downstream ON vs OFF firing pattern)")
print(f"{'='*60}")
print(f"{'Condition':<30} {'Mean Ham.':>10} {'95% CI':>18} {'p-value':>10}")
print("-" * 72)
print(f"{'Biological':<30} {ham_bio:>10.3f} {'—':>18} {'—':>10}")
for label, arr in [('Rewired', ham_rw), ('Shuffled', ham_sh), ('Random', ham_rand)]:
    lo, hi = bootstrap_ci(arr)
    p_val  = permutation_p(arr, ham_bio)
    sig    = '✅ p<0.05' if p_val < 0.05 else '⚠️  n.s.'
    print(f"  {label:<28} {np.mean(arr):>10.3f} [{lo:.3f}, {hi:.3f}]  {p_val:>8.4f}  {sig}")

# ── V1 FIX: Violin + box + scatter panels ────────────────────────────────
with plt.style.context('dark_background'):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: Trial scatter
    ax = axes[0]
    ax.axhline(ham_bio, color='#FFD700', lw=2.5, ls='--',
               label=f'Bio={ham_bio:.3f}')
    for data, color, marker, lbl in [
        (ham_rw,   '#4A90D9', 'o', 'Rewired'),
        (ham_sh,   '#2ECC71', 's', 'Shuffled'),
        (ham_rand, '#E8893C', '^', 'Random'),
    ]:
        ax.scatter(range(N_TRIALS), data, color=color, s=45, label=lbl,
                   marker=marker, zorder=5)
    ax.set_ylim(-0.05, 1.15)
    ax.set_xlabel('Trial'); ax.set_ylabel('Downstream DSI')
    ax.set_title('Trial-by-trial Hamming distance\n(downstream ON vs OFF pattern)', color='#FFD700', fontsize=10)
    ax.legend(fontsize=8); ax.grid(alpha=0.15)

    # Panel 2: Violin plot (V1 NEW)
    ax = axes[1]
    # Use Hamming distance (primary metric) for violin/box plots
    data_vl  = [ham_rw, ham_sh, ham_rand]
    vp = ax.violinplot(data_vl, positions=[1, 2, 3], showmedians=True)
    for i, (pc, color) in enumerate(zip(vp['bodies'],
                                         ['#4A90D9','#2ECC71','#E8893C'])):
        pc.set_facecolor(color)
        pc.set_alpha(0.6)
    ax.axhline(ham_bio, color='#FFD700', lw=2.5, ls='--',
               label=f'Bio={ham_bio:.3f}')
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(['Rewired', 'Shuffled', 'Random'], fontsize=9)
    ax.set_ylabel('Downstream Hamming distance')
    ax.set_title('Violin: Hamming Distribution\nvs. Biological Baseline', color='#FFD700', fontsize=10)
    ax.legend(fontsize=8); ax.grid(alpha=0.15, axis='y')

    # Panel 3: Box plot + bootstrap CI
    ax = axes[2]
    bp = ax.boxplot(data_vl, patch_artist=True,
                    boxprops=dict(facecolor='#333', color='white'),
                    medianprops=dict(color='#FFD700', lw=2),
                    whiskerprops=dict(color='white'),
                    capprops=dict(color='white'),
                    flierprops=dict(color='white', marker='o'))
    for patch, color in zip(bp['boxes'], ['#4A90D9','#2ECC71','#E8893C']):
        patch.set_facecolor(color + '88')
    means = [np.mean(d) for d in data_vl]
    cis   = [bootstrap_ci(d) for d in data_vl]
    errs  = [[m - lo for m, (lo, _)  in zip(means, cis)],
             [hi - m for m, (_, hi) in zip(means, cis)]]
    ax.errorbar([1, 2, 3], means, yerr=errs, fmt='none',
                color='white', capsize=6, lw=2, zorder=5)
    ax.axhline(ham_bio, color='#FFD700', lw=2.5, ls='--',
               label=f'Bio={ham_bio:.3f}')
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(['Rewired', 'Shuffled', 'Random'], fontsize=9)
    ax.set_ylabel('Downstream Hamming distance')
    ax.set_title('Box + Bootstrap 95% CI\n(Hamming distance)', color='#FFD700', fontsize=10)
    ax.legend(fontsize=8); ax.grid(alpha=0.15, axis='y')

    plt.suptitle('BioWire V1 — Three-Layer Negative Control  (Hamming distance, downstream)',
                 fontsize=13, color='#FFD700')
    plt.tight_layout()
    plt.savefig('biowire_v1_negative_control.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
    plt.show()

print("\nSaved → biowire_v1_negative_control.png")

## Summary — BioWire v1.0.0

### Simulation Results (real FlyWire connectome, seed=42)

| Metric | Value |
|--------|-------|
| Neuron types modelled | 100 (ME\_L) |
| Synaptic edges | 308 |
| Calibration (ephys target) | **5 / 8 neurons in range** |
| Grid search best score | 14.80 |
| Best params | G\_on=0.33, G\_off=0.15, W\_scale=0.06 |
| DSI (ON-only vs OFF-only) | 1.00 (reflects sparse 100-type network) |
| Null stimulus silence | ✅ confirmed |
| Negative control (Random Hamming) | 0.003 ± 0.004, p = 0.057 (n.s.) |

### Architecture

| Component | Implementation |
|-----------|---------------|
| Connectome source | FlyWire ME\_L |
| Offline mode | Synthetic mock connectome auto-generated |
| Weight format | scipy.sparse CSR → 4-bit signed integer |
| Simulator | Brian2, conductance-based LIF, dt=0.5 ms |
| E/I classification | 30 inhibitory types (DOI-cited) |
| Parameter search | Grid search, calibrated to in-vivo ephys |
| Negative control | Degree-preserving rewire · weight shuffle · random |
| Visualisation | Dark-background matplotlib, 6 figures saved |

### References

- Clark et al. (2011). *Defining the computational structure of the motion detector in Drosophila.* Neuron. doi:[10.1016/j.neuron.2011.06.004](https://doi.org/10.1016/j.neuron.2011.06.004)
- Takemura et al. (2017). *Connectome of the fly visual circuitry.* eLife. doi:[10.7554/eLife.24394](https://doi.org/10.7554/eLife.24394)
- Strother et al. (2017). *The emergence of directional selectivity in the visual motion pathway of Drosophila.* Neuron. doi:[10.1016/j.neuron.2017.07.025](https://doi.org/10.1016/j.neuron.2017.07.025)
- Shinomiya et al. (2019). *Comparisons between the ON- and OFF-edge motion pathways in the Drosophila visual system.* eLife. doi:[10.7554/eLife.37833](https://doi.org/10.7554/eLife.37833)
- Maisak et al. (2013). *A directional tuning map of Drosophila elementary motion detectors.* Nature. doi:[10.1038/nature12320](https://doi.org/10.1038/nature12320)
- Eckstein et al. (2024). *Neurotransmitter classification from electron microscopy images at synaptic sites in Drosophila.* Nature. doi:[10.1038/s41586-024-07558-y](https://doi.org/10.1038/s41586-024-07558-y)
- Dorkenwald et al. (2024). *FlyWire: online community for whole-brain connectomics.* Nature Methods. doi:[10.1038/s41592-024-02237-2](https://doi.org/10.1038/s41592-024-02237-2)
- Nern et al. (2015). *Quantitative neuroanatomy for connectomics in Drosophila.* eLife. doi:[10.7554/eLife.04693](https://doi.org/10.7554/eLife.04693)
